In [1]:
import torch
from snac import SNAC
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

import sys
sys.path.append("/home/dkham/Documents/year4/ar-bird-vocalization/")

/home/dkham/Documents/year4/ar-bird-vocalization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
Audio("/home/dkham/.cache/huggingface/datasets/downloads/extracted/XCM_shard_0131/XC776755.ogg")

In [2]:
full_dataset = load_dataset(
        "DBD-research-group/BirdSet",
        "XCM",
        split="train",
        trust_remote_code=True,
    )

In [3]:
import librosa

model = SNAC.from_pretrained("hubertsiuzdak/snac_32khz").eval().cuda()
snac_sr = 32000

info = sf.info(full_dataset[0]["filepath"])
native_sr = info.samplerate
audio, sr = sf.read(
    full_dataset[0]["filepath"],
    start=int(2.64 * native_sr),
    stop=int(5.264 * native_sr),
)
if sr != snac_sr:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=snac_sr)
audio = torch.tensor(audio, dtype=torch.float32).reshape(1, 1, -1).cuda()

with torch.inference_mode():
    codes = model.encode(audio)
    audio_hat = model.decode(codes)

In [4]:
codes[0].shape, codes[1].shape, codes[2].shape, codes[3].shape

(torch.Size([1, 28]),
 torch.Size([1, 56]),
 torch.Size([1, 112]),
 torch.Size([1, 224]))

In [5]:
audio_hat.shape, audio.shape

(torch.Size([1, 1, 86016]), torch.Size([1, 1, 83968]))

In [6]:
audio_hat = audio_hat[:, :, :audio.shape[-1]]

In [7]:
audio_hat.shape

torch.Size([1, 1, 83968])

In [8]:
audio_hat.cpu().numpy().flatten().shape

(83968,)

In [9]:
Audio(audio_hat.cpu().numpy().flatten(), rate=sr)

In [10]:
Audio(audio.cpu().numpy().flatten(), rate=sr)

In [11]:
from snac_inference import flatten_codes, unflatten_codes
flattened = flatten_codes(codes)

In [12]:
flattened

array([[ 3855,  5079,  5303,  9050, 11660, 11388, 11564, 14391, 14019,
        13773, 14368, 14104, 14038, 14219, 12555,  2785,  5889,  7718,
        10156,  8237, 11744, 10570, 15169, 15995, 13978, 15454, 12764,
        13332, 14609, 15162,  3380,  4292,  4886,  9233, 10206,  9519,
        11131, 15349, 12689, 13240, 13145, 13392, 15303, 14511, 14430,
         3157,  6292,  4509, 11640,  9836, 10974, 10165, 12675, 14901,
        15902, 15111, 16250, 13981, 16258, 15705,  3157,  6996,  4171,
        10847, 11546, 10576,  9249, 12309, 13815, 13933, 12641, 13309,
        13194, 12481, 13433,  2109,  4512,  4723, 10569,  9567, 12155,
         8861, 12381, 12634, 12536, 15263, 14055, 13031, 12781, 13209,
         2109,  6572,  4432,  8909,  9567,  8504,  8583, 13027, 12342,
        15882, 14678, 14480, 12623, 13913, 13194,  1065,  6451,  6832,
        10301,  9413, 10673,  8986, 15156, 15036, 14602, 12630, 13859,
        14561, 16349, 15882,  4052,  4493,  7155, 11108, 11005, 10332,
      

In [13]:
codes

[tensor([[3855, 2785, 3380, 3157, 3157, 2109, 2109, 1065, 4052, 3601, 1566, 3695,
          3335, 2976, 2783, 2111, 4052, 2738, 3695, 1239, 4052, 2785, 1539, 3993,
          2738, 1239, 3035, 1186]], device='cuda:0'),
 tensor([[ 983, 1207, 1793, 3622,  196,  790, 2196,  413, 2900,   75,  416,  627,
          2476,  336, 2355, 2736,  397, 3059, 2407,  621,  252, 3619, 1306, 1204,
          3967, 1306, 1712,  638,  445, 3713,  974,  887,  613,  979,  621, 3488,
          1496,  121, 4034,  361,  627, 3249, 2552, 1032,  992, 2341,  445, 1195,
          1782, 3622, 1581,  863, 1035, 3811, 1208, 1808]], device='cuda:0'),
 tensor([[ 858, 3468, 3196, 3372, 1964,   45, 3552, 2378, 1041, 2014, 1327, 2939,
          3448, 1644, 2782, 1973, 2655, 3354, 2384, 1057, 2377, 1375, 3963,  669,
           717, 1375,  312,  391, 2109, 1221, 2481,  794, 2916, 2813, 2140, 1644,
          2982, 2064, 2398, 1463, 1525,   38,  671,  923, 1605,    0, 2840, 1602,
          1329, 1644, 2867,  409, 1443, 1396, 37

In [14]:
unflattened = unflatten_codes(flattened[0])

In [15]:
unflattened

[array([3855, 2785, 3380, 3157, 3157, 2109, 2109, 1065, 4052, 3601, 1566,
        3695, 3335, 2976, 2783, 2111, 4052, 2738, 3695, 1239, 4052, 2785,
        1539, 3993, 2738, 1239, 3035, 1186]),
 array([ 983, 1207, 1793, 3622,  196,  790, 2196,  413, 2900,   75,  416,
         627, 2476,  336, 2355, 2736,  397, 3059, 2407,  621,  252, 3619,
        1306, 1204, 3967, 1306, 1712,  638,  445, 3713,  974,  887,  613,
         979,  621, 3488, 1496,  121, 4034,  361,  627, 3249, 2552, 1032,
         992, 2341,  445, 1195, 1782, 3622, 1581,  863, 1035, 3811, 1208,
        1808]),
 array([ 858, 3468, 3196, 3372, 1964,   45, 3552, 2378, 1041, 2014, 1327,
        2939, 3448, 1644, 2782, 1973, 2655, 3354, 2384, 1057, 2377, 1375,
        3963,  669,  717, 1375,  312,  391, 2109, 1221, 2481,  794, 2916,
        2813, 2140, 1644, 2982, 2064, 2398, 1463, 1525,   38,  671,  923,
        1605,    0, 2840, 1602, 1329, 1644, 2867,  409, 1443, 1396, 3776,
        2220, 2974, 2561, 1210, 2157, 3660, 1463, 